In [1]:
import random,os
import pandas as pd
import numpy as np
import missingno as msno
from Data_cleaning_utils import perform_data_cleaning
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,KNNImputer,IterativeImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn import set_config
set_config(transform_output='pandas')
import optuna as optuna


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed = 0
seed_everything(seed)

import dagshub
dagshub.init(repo_owner='shapniljoy', repo_name='delivery-time-prediction', mlflow=True)

import mlflow
mlflow.set_tracking_uri("https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow")
mlflow.set_experiment('Final Model')

Accessing as shapniljoy

Initialized MLflow to track repo "shapniljoy/delivery-time-prediction"

Repository shapniljoy/delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/f6d08d54978041efba3e88cb266b6058', creation_time=1782115728966, experiment_id='7', last_update_time=1782115728966, lifecycle_stage='active', name='Final Model', tags={}, trace_location=None, workspace='default'>

In [2]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)
df = df.dropna()

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','age',
        'ratings','distance_km','pickup_time','day','month'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (30156, 15) and (30156,) 

Testing data: (7539, 15) and (7539,) 



In [3]:
rating_order = ['less than 4','4-4.5','4.5-5']

distance_order = ['short','medium','long','very_long']

city_type_order = ['Semi-Urban','Urban','Metropolitan']

pickup_time_order = ['5 minutes','10 minutes','15 minutes']

ordinal_encoding = OrdinalEncoder(categories=[rating_order,distance_order,city_type_order,pickup_time_order],
                                  handle_unknown='use_encoded_value', unknown_value=-999)


preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['ratings_cat','distance_km_cat','city_type','pickup_time_cat']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'age_cat','weather','traffic','festival','time_of_day']),
        ],remainder='passthrough')

In [4]:
lgbm_params = {
 'n_estimators': 1940,
 'max_depth': 7,
 'num_leaves': 57,
 'min_child_samples': 228,
 'colsample_bytree': 0.8452545147323064,
 'learning_rate': 0.046640660322683554,
 'reg_alpha': 0.020341454480246005,
 'reg_lambda': 0.10776961230711216,
 'boosting_type': 'dart',
 'subsample': 0.9481361810641186
}

RF_params = {
    'n_estimators': 641,
    'max_depth': 13,
    'min_samples_split': 6,
    'min_samples_leaf': 6,
    'max_features': None,
    'max_samples': 0.992036436296416
}

Lasso_params = {
    'alpha': 0.2606044403283886
}

lgbm = LGBMRegressor(**lgbm_params,verbose=-1)
rf = RandomForestRegressor(**RF_params)
lasso = Lasso(**Lasso_params)

In [5]:
with mlflow.start_run(run_name='Final Model') as parent:

    model = StackingRegressor(estimators=[('lgbm',lgbm),('rf',rf)],
                                    final_estimator=lasso,verbose=False)

    model_pipe = Pipeline([
        ('preprocessor',preprocessor),
        ('model',model)
    ])

    final_model = model_pipe.fit(x_train,y_train)

    y_pred_train = final_model.predict(x_train)
    y_pred_test = final_model.predict(x_test)

    print(f"Training error: {mean_absolute_error(y_train,y_pred_train)}")
    print(f"Testing error: {mean_absolute_error(y_test,y_pred_test)}")
    print(f"Training R2 score: {r2_score(y_train,y_pred_train)}")
    print(f"Testing R2 score: {r2_score(y_test,y_pred_test)}")

    mlflow.log_metric('Training error',mean_absolute_error(y_train,y_pred_train))
    mlflow.log_metric('Testing error',mean_absolute_error(y_test,y_pred_test))
    mlflow.log_metric('Training R2 score',r2_score(y_train,y_pred_train))
    mlflow.log_metric('Testing R2 score',r2_score(y_test,y_pred_test))

    mlflow.sklearn.log_model(final_model,name='Final Model')


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but Lasso was fitted without feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but Lasso was fitted without feature names
  warnings.warn(


Training error: 3.3187787702520635
Testing error: 3.5131626105648444
Training R2 score: 0.7950227207500729
Testing R2 score: 0.7733869964977694


2026/06/22 14:14:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Final Model at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/7/runs/5512fe1a1ea04a00aac439162bfb5146
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/7


# Optuna after dropping missing values and With Numerical columns

In [2]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)
df = df.dropna()

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','day','month',
        'age_cat','ratings_cat','distance_km_cat','pickup_time_cat'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (30156, 15) and (30156,) 

Testing data: (7539, 15) and (7539,) 



In [3]:

numerical_cols = ['age','ratings','distance_km','pickup_time']

city_type_order = ['Semi-Urban','Urban','Metropolitan']

ordinal_encoding = OrdinalEncoder(categories=[city_type_order],
                                  handle_unknown='use_encoded_value', unknown_value=-999)


preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['city_type']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'weather','traffic','festival','time_of_day']),
        ('scaling',StandardScaler(),numerical_cols),
        ],remainder='passthrough')

In [4]:
catboost_params = {'iterations': 618, 
                  'depth': 7, 
                  'learning_rate': 0.029550509331092316, 
                  'l2_leaf_reg': 31.153463513382516, 
                  'random_strength': 1, 
                  'bagging_temperature': 0.7163272041185655, 
                  'border_count': 96, 
                  'grow_policy': 'Depthwise', 
                  'max_ctr_complexity': 7}

RF_params = {'n_estimators': 902,
            'max_depth': 14,
            'min_samples_split': 10,
            'min_samples_leaf': 7,
            'max_features': None,
            'max_samples': 0.9857978714826628}

lasso_params = {'alpha': 0.23980657464134286}


catboost = CatBoostRegressor(**catboost_params,verbose=False)
rf = RandomForestRegressor(**RF_params,bootstrap=True,random_state=seed,verbose=False)
lasso = Lasso(**lasso_params)

In [5]:
with mlflow.start_run(run_name='Final Model') as parent:

    model = StackingRegressor(estimators=[('catboost',catboost),('rf',rf)],
                                    final_estimator=lasso,verbose=False)

    model_pipe = Pipeline([
        ('preprocessor',preprocessor),
        ('model',model)
    ])

    final_model = model_pipe.fit(x_train,y_train)

    y_pred_train = final_model.predict(x_train)
    y_pred_test = final_model.predict(x_test)

    print(f"Training error: {mean_absolute_error(y_train,y_pred_train)}")
    print(f"Testing error: {mean_absolute_error(y_test,y_pred_test)}")
    print(f"Training R2 score: {r2_score(y_train,y_pred_train)}")
    print(f"Testing R2 score: {r2_score(y_test,y_pred_test)}")

    mlflow.log_metric('Training error',mean_absolute_error(y_train,y_pred_train))
    mlflow.log_metric('Testing error',mean_absolute_error(y_test,y_pred_test))
    mlflow.log_metric('Training R2 score',r2_score(y_train,y_pred_train))
    mlflow.log_metric('Testing R2 score',r2_score(y_test,y_pred_test))

    mlflow.sklearn.log_model(final_model,name='Final Model with numerical cols')


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but Lasso was fitted without feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but Lasso was fitted without feature names
  warnings.warn(


Training error: 2.68587641380368
Testing error: 3.0168762962799747
Training R2 score: 0.8732631368851098
Testing R2 score: 0.839841554001954


2026/06/22 22:09:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Final Model at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/7/runs/8187c4070aed452db658b450e6d6e0fc
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/7
